# Convert Spatial Matrices to Long Format

This notebook converts harmonized wide spatial matrices into the long-format CSVs expected by the website.

The source files currently contain log2 expression values (not z-scores).

Run the cells in order. The output files are written to the same local folder as the RNA source by default.

In [15]:
from pathlib import Path
import csv

RNA_DATA_DIR = Path(r"C:\Users\wilke\OneDrive - Hubrecht Institute\2025-Proteomics\Proteomics_data\RNAbulkseq\Spatial\hub-ks-b004-20230825\rerun-2026")
PROT_DATA_DIR = Path(r"C:\Users\wilke\OneDrive - Hubrecht Institute\2025-Proteomics\Proteomics_data\20260317_spatial_astral\20260326_adjustedimputation")


OUTPUT_DIR = Path(r"C:\Users\wilke\OneDrive - Hubrecht Institute\2025-Proteomics\Proteomics_data\20260317_spatial_astral\Wilke\Datafiles")
RNA_INPUT = RNA_DATA_DIR / "normalized_counts_20260528_harmonized.txt"
PROT_INPUT = PROT_DATA_DIR / "dep_spatial_assay_20260528_harmonized.txt"
RNA_OUTPUT = OUTPUT_DIR / "rna_long_harmonized_20260528.csv"
PROT_OUTPUT = OUTPUT_DIR / "prot_long_harmonized_20260528.csv"

print("RNA input folder:", RNA_DATA_DIR)
print("Protein input folder:", PROT_DATA_DIR)
print("RNA input exists:", RNA_INPUT.exists(), RNA_INPUT)
print("Protein input exists:", PROT_INPUT.exists(), PROT_INPUT)
print("RNA output:", RNA_OUTPUT)
print("Protein output:", PROT_OUTPUT)

RNA input folder: C:\Users\wilke\OneDrive - Hubrecht Institute\2025-Proteomics\Proteomics_data\RNAbulkseq\Spatial\hub-ks-b004-20230825\rerun-2026
Protein input folder: C:\Users\wilke\OneDrive - Hubrecht Institute\2025-Proteomics\Proteomics_data\20260317_spatial_astral\20260326_adjustedimputation
RNA input exists: True C:\Users\wilke\OneDrive - Hubrecht Institute\2025-Proteomics\Proteomics_data\RNAbulkseq\Spatial\hub-ks-b004-20230825\rerun-2026\normalized_counts_20260528_harmonized.txt
Protein input exists: True C:\Users\wilke\OneDrive - Hubrecht Institute\2025-Proteomics\Proteomics_data\20260317_spatial_astral\20260326_adjustedimputation\dep_spatial_assay_20260528_harmonized.txt
RNA output: C:\Users\wilke\OneDrive - Hubrecht Institute\2025-Proteomics\Proteomics_data\20260317_spatial_astral\Wilke\Datafiles\rna_long_harmonized_20260528.csv
Protein output: C:\Users\wilke\OneDrive - Hubrecht Institute\2025-Proteomics\Proteomics_data\20260317_spatial_astral\Wilke\Datafiles\prot_long_harmoni

In [17]:
import math

GROUP_BY_PREFIX = {
    "P": "Posterior",
    "A": "Anterior",
    "S": "Somite",
}

ROMAN_TO_INT = {
    "I": 1,
    "II": 2,
    "III": 3,
    "IV": 4,
    "V": 5,
    "VI": 6,
}

def infer_group(sample_name: str) -> str:
    sample_name = (sample_name or "").strip()
    if not sample_name:
        return ""
    return GROUP_BY_PREFIX.get(sample_name[0].upper(), "")

def infer_replicate(sample_name: str) -> str:
    sample_name = (sample_name or "").strip()
    if not sample_name or "-" not in sample_name:
        return ""

    suffix = sample_name.split("-", 1)[1].strip().upper()
    if suffix in ROMAN_TO_INT:
        return str(ROMAN_TO_INT[suffix])
    if suffix.isdigit():
        return suffix
    return ""

def infer_group_from_repl_column(column_name: str) -> str:
    key = (column_name or "").strip().split("_", 1)[0].upper()
    return GROUP_BY_PREFIX.get(key, "")

def infer_replicate_from_column(column_name: str) -> str:
    parts = (column_name or "").strip().split("_")
    if not parts:
        return ""
    tail = parts[-1]
    digits = "".join(ch for ch in tail if ch.isdigit())
    return digits

def convert_rna_matrix_to_long(input_path: Path, output_path: Path) -> int:
    with input_path.open("r", newline="", encoding="utf-8-sig") as infile:
        reader = csv.reader(infile, delimiter="\t")
        header = next(reader, None)
        if not header:
            raise ValueError(f"No header found in {input_path}")

        if "sample" not in header:
            raise ValueError(
                f"Expected a 'sample' column in {input_path}, found: {', '.join(header[:10])}"
            )
        sample_idx = header.index("sample")

        gene_columns = []
        for idx, col in enumerate(header):
            clean = (col or "").strip()
            if idx == sample_idx or not clean:
                continue
            gene_columns.append((idx, clean))

        output_path.parent.mkdir(parents=True, exist_ok=True)
        row_count = 0
        with output_path.open("w", newline="", encoding="utf-8") as outfile:
            writer = csv.DictWriter(
                outfile,
                fieldnames=["sample", "Gene", "value", "group", "replicate"],
            )
            writer.writeheader()

            for row in reader:
                if not row:
                    continue

                sample_name = (row[sample_idx] if sample_idx < len(row) else "").strip()
                if not sample_name:
                    continue

                group = infer_group(sample_name)
                replicate = infer_replicate(sample_name)

                for col_idx, gene_name in gene_columns:
                    value = row[col_idx] if col_idx < len(row) else ""
                    if value in (None, ""):
                        continue

                    try:
                        numeric_value = float(value)
                    except ValueError as exc:
                        raise ValueError(
                            f"Non-numeric RNA value for gene '{gene_name}' in sample '{sample_name}': {value}"
                        ) from exc

                    if numeric_value < 0:
                        raise ValueError(
                            f"Negative RNA value for gene '{gene_name}' in sample '{sample_name}': {numeric_value}"
                        )

                    # RNA source is not log-transformed yet; apply log2(x+1) once here.
                    log2_value = math.log2(numeric_value + 1.0)

                    writer.writerow({
                        "sample": sample_name,
                        "Gene": gene_name,
                        "value": log2_value,
                        "group": group,
                        "replicate": replicate,
                    })
                    row_count += 1

        return row_count

def convert_protein_matrix_to_long(input_path: Path, output_path: Path) -> int:
    with input_path.open("r", newline="", encoding="utf-8-sig") as infile:
        reader = csv.DictReader(infile, delimiter="\t")
        if not reader.fieldnames:
            raise ValueError(f"No header found in {input_path}")

        gene_field = "gene" if "gene" in reader.fieldnames else None
        if gene_field is None:
            raise ValueError(
                f"Expected a 'gene' column in {input_path}, found: {', '.join(reader.fieldnames[:10])}"
            )

        repl_columns = [
            name for name in reader.fieldnames
            if name != gene_field and name and "repl" in name.lower()
        ]
        if not repl_columns:
            raise ValueError(f"No replicate columns found in {input_path}")

        output_path.parent.mkdir(parents=True, exist_ok=True)
        row_count = 0
        with output_path.open("w", newline="", encoding="utf-8") as outfile:
            writer = csv.DictWriter(
                outfile,
                fieldnames=["sample", "Gene", "value", "group", "replicate"],
            )
            writer.writeheader()

            for row in reader:
                gene_name = (row.get(gene_field) or "").strip()
                if not gene_name:
                    continue

                for column_name in repl_columns:
                    value = row.get(column_name)
                    if value in (None, ""):
                        continue
                    writer.writerow({
                        "sample": column_name,
                        "Gene": gene_name,
                        "value": value,
                        "group": infer_group_from_repl_column(column_name),
                        "replicate": infer_replicate_from_column(column_name),
                    })
                    row_count += 1

        return row_count

In [18]:
rna_rows = convert_rna_matrix_to_long(RNA_INPUT, RNA_OUTPUT)
prot_rows = convert_protein_matrix_to_long(PROT_INPUT, PROT_OUTPUT)

print(f"Wrote {rna_rows:,} RNA rows to {RNA_OUTPUT}")
print(f"Wrote {prot_rows:,} protein rows to {PROT_OUTPUT}")

Wrote 164,619 RNA rows to C:\Users\wilke\OneDrive - Hubrecht Institute\2025-Proteomics\Proteomics_data\20260317_spatial_astral\Wilke\Datafiles\rna_long_harmonized_20260528.csv
Wrote 91,773 protein rows to C:\Users\wilke\OneDrive - Hubrecht Institute\2025-Proteomics\Proteomics_data\20260317_spatial_astral\Wilke\Datafiles\prot_long_harmonized_20260528.csv


In [19]:
def preview_csv(path: Path, limit: int = 5):
    with path.open("r", encoding="utf-8") as handle:
        for index, line in enumerate(handle):
            print(line.rstrip())
            if index + 1 >= limit:
                break

print("RNA preview:")
preview_csv(RNA_OUTPUT)
print()
print("Protein preview:")
preview_csv(PROT_OUTPUT)

RNA preview:
sample,Gene,value,group,replicate
P-I,Xkr4,2.560040056742831,Posterior,1
P-I,Gm19938,3.061060876796761,Posterior,1
P-I,Cyp4a10,1.786019294989321,Posterior,1
P-I,Sox17,7.547628296340262,Posterior,1

Protein preview:
sample,Gene,value,group,replicate
A_repl1,1110065P20Rik,22.7617116966596,Anterior,1
A_repl2,1110065P20Rik,22.5524493203229,Anterior,2
A_repl3,1110065P20Rik,22.1513423932751,Anterior,3
P_repl1,1110065P20Rik,22.181595215282,Posterior,1


## Next step

After the two long-format CSV files are created, upload them to GitHub. Once they are in the repo or your data repository, the website can be pointed directly at those files instead of reshaping the wide matrices in the browser.